# Feature Attribution using Ranking - v1.0

In [1]:
import sys

!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install catboost
!{sys.executable} -m pip install nbimporter

In [2]:
import time
import numpy as np
import pandas as pd
import zipfile as zf
import nbimporter

import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
from catboost import CatBoostClassifier

from scipy.sparse.linalg import eigs

import networkx as nx
import GraphPR as gpr
import PageRank as pr

# Data preprocessing methods

In [3]:
# return one df without nan's on num_cols
# numerical features: fill nan's with median/mean/mode
def pre_proc_fillna_num_fts(df,num_cols,num_type='mean'):
    df_train= df.copy()

    if(num_type=='median'):
        for col in num_cols:
            ft_median= df_train[col].median()
            df_train[col]= df_train[col].fillna(ft_median)
    elif(num_type=='mode'):
        for col in num_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)
    else:
        for col in num_cols:
            ft_mean= df_train[col].mean()
            df_train[col]= df_train[col].fillna(ft_mean)

    return df_train

In [4]:
# return one df without nan's on cat_cols
# categorical features: fill nan's with mode/mean/median
def pre_proc_fillna_cat_fts(df,cat_cols,cat_type='mode'):
    df_train= df.copy()
    
    if(cat_type!='mode' and type(df_train[cat_cols[0]].value_counts().index[0])!=type('str')):
        if(cat_type=='mean'):
            for col in cat_cols:
                ft_mean= df_train[col].mean()
                df_train[col]= df_train[col].fillna(ft_mean)
        elif(cat_type=='median'):
            for col in cat_cols:
                ft_median= df_train[col].median()
                df_train[col]= df_train[col].fillna(ft_median)
    else:
        for col in cat_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)

    return df_train

In [5]:
# n_cols refers only to df's columns with numerical values
def normalize_selected_cols(df, n_cols):
    result= df.copy()
    
    for col in n_cols:
        max_value= df[col].max()
        min_value= df[col].min()
        result[col]= (df[col]- min_value)/ (max_value - min_value)
        
    return result

In [18]:
# return a sorted DataFrame with Features and Importances - OHE compacted, that is, dataset's original features
def ft_importance_df(importances, ft_names, replace_list):
    fti= pd.Series(importances, index=ft_names).sort_values(ascending=False).to_frame().reset_index()
    fti= fti.rename(columns= {'index':'Feature',0:'Importance'}, inplace=False)
    fti['Feature'].replace(replace_list, inplace=True)
    fti= fti.groupby(['Feature']).sum().sort_values('Importance', ascending=False).reset_index()
    
    return fti

# Feature Attribution using Ranking methods

In [6]:
# return a DataFrame with replace values (mean/median/mode/none to categorical and numeric) to fill train cols. df is post-processed (cat_cols encoded)
def replace_values(df,num_cols,num_type='mean',cat_type='none'):
    cat_values= None
    num_values= None
    
    if (cat_type=='mean'):
        cat_values= df.mean(axis=0).to_frame().T
    elif (cat_type=='median'):
        cat_values= df.median(axis=0).to_frame().T
    elif (cat_type=='mode'):
        cat_values= df.mode(axis=0)
    
    if (num_type=='mode'):
        num_values= df.mode(axis=0)
    elif (num_type=='median'):
        num_values= df.median(axis=0).to_frame().T
    elif (num_type=='mean'):
        num_values= df.mean(axis=0).to_frame().T

    if(cat_type!='none'):
        cat_values[num_cols]= num_values[num_cols]
        return cat_values
    
    return num_values

In [117]:
# train the ML model and return its mean accuracy after n_train runs
def train_model_get_acc_mean(model, x_trn, x_tst, y_trn, y_tst, n_train):
    trainings= []
    
    np_ar= np.ndarray((0,0))    # model.fit() expect a 1d array instead of a pd.DataFrame
    pd_df= pd.DataFrame(np_ar)  # we need to verify and change it using values.ravel()
    
    if (type(y_trn)== type(pd_df)):
        
        for i in range(n_train):

            model.fit(x_trn, y_trn.values.ravel())
            acc= sklearn.metrics.accuracy_score(y_tst, model.predict(x_tst))

            trainings.append(acc)
    else:
        for i in range(n_train):

            model.fit(x_trn, y_trn)
            acc= sklearn.metrics.accuracy_score(y_tst, model.predict(x_tst))

            trainings.append(acc)
            
    return np.mean(trainings)

In [8]:
# re-training is needed because machine learning models typically assume that the train and the test data comes from a similar distribution 
# (Hooker et al., 2018)
# here we return p(x|i) and p(x|ij)
def remove_and_retrain_v1(model, replace_ft_vals, x_trn, x_tst, y_trn, y_tst, n_train, verbose=False):
    acc_no_i= []
    acc_no_ij= []

    n_fts= len(x_trn.columns)

    start= time.time()

    for i in range(n_fts):
        acc_row= []

        # replace the i-th ft with its respective mode/mean to "remove" it. train the ML model and get the mean accuracy
        train_copy_no_i= x_trn.copy()
        train_copy_no_i.loc[:,train_copy_no_i.columns[i]]= replace_ft_vals.iloc[0,i]

        acc_no_i.append(train_model_get_acc_mean(model, train_copy_no_i, x_tst, y_trn, y_tst, n_train))

        for j in range(n_fts):

            if (i!= j):
                # replace the j-th ft with its respective mode/mean to "remove" it. here, we "remove" the i-th and the j-th ft
                # train the ML model and get the mean accuracy
                train_copy_no_ij= train_copy_no_i.copy()
                train_copy_no_ij.loc[:,train_copy_no_ij.columns[j]]= replace_ft_vals.iloc[0,j]

                acc_row.append(train_model_get_acc_mean(model, train_copy_no_ij, x_tst, y_trn, y_tst, n_train))
            else:
                acc_row.append(0)

        acc_no_ij.append(acc_row)

    end= time.time()
    
    if (verbose==True):
        print("--- %s seconds ---" % np.round((end- start), 2))

    return acc_no_i, acc_no_ij

In [9]:
# here we return | p(x|ij) - p(x|i) |
def get_p_matrix_v1(n_fts, acc_no_i, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pi)
                
    return p_matrix

In [10]:
# here we return | p(x|ij) - p(x|j) |
def get_p_matrix_v2(n_fts, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pj)
                
    return p_matrix

In [11]:
# here we return (| p(x|ij) - p(x|j) | + | p(x|i) - p(x) |) / 2
def get_p_matrix_v3(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pj)+ abs(pi- p))/ 2
                
    return p_matrix

In [12]:
# here we return (| p(x|ij) - p(x|i) | + | p(x|j) - p(x) |) / 2
def get_p_matrix_v4(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pi)+ abs(pj- p))/ 2
                
    return p_matrix

In [13]:
# the stationary distribution is the fraction of time that the system spends in each state as the number of samples approaches infinity
# it looks like there's not a built-in method to find the stationary distribution

# converts a matrix to a row stochastic matrix - a real square matrix, with each row summing to 1
def to_row_stochastic_matrix(M):
    result= M
    
    for row in result:
        n= sum(row)
        if n> 0:
            row[:]= [f/sum(row) for f in row]
    
    return result

In [14]:
# the stationary distribution - analytical solution
# return 1D array
def stationary_dist_v1(stochastic_matrix):
    
    size_A= stochastic_matrix.shape[1]
    ones= [1]* size_A

    A= np.append(np.transpose(stochastic_matrix)- np.identity(size_A),[ones],axis=0)

    v= np.zeros(size_A+ 1)
    v[size_A]= 1
    v= np.transpose(v)

    stationary= np.linalg.solve(np.transpose(A).dot(A), np.transpose(A).dot(v))

    return stationary

In [15]:
# the stationary distribution - another analytical solution
# return 2D array
def stationary_dist_v2(stochastic_matrix):
    # we have to transpose so that Markov transitions correspond to right multiplying by a column vector
    eigval, eigvec= eigs(stochastic_matrix.T, k=1, which='LM')
    stationary= eigvec/ eigvec.sum()

    # eigs finds complex eigenvalues and eigenvectors, so you'll want the real part.
    stationary= stationary.real

    return stationary

In [16]:
# convert a p_matrix to a right stochastic matrix - a real square matrix, with each row summing to 1
def p_matrix_to_row_stochastic_matrix(p_matrix):
    
    st_matrix= np.asarray(p_matrix)
    
    # now convert to right stochastic matrix
    st_matrix= to_row_stochastic_matrix(st_matrix)
    
    return st_matrix

In [17]:
# find the stationary distribution
# return 1D array (version=1) or 2D array (version=1)
def p_matrix_to_stationary_dist(p_matrix, version=1):
    
    st_matrix= np.asarray(p_matrix)
    
    # now convert to right stochastic matrix
    st_matrix= to_row_stochastic_matrix(st_matrix)
    
    # then get the stationary distribution
    stationary_d= []
    
    if (version==1):
        stationary_d= stationary_dist_v1(st_matrix)
    else:
        stationary_d= stationary_dist_v2(st_matrix)
    
    return stationary_d

In [19]:
import math
from sklearn.neighbors import NearestNeighbors

# find the test_size nearest neighbors from a target_X instance
# return train, test, labels_train, labels_test datsets based on knn, with target_X and target_Y into testX and testY
def knn_train_test_split(df_X, df_Y, target_X, target_Y, test_size= 0.2):
    
    m_ins= df_X.shape[0]

    neighbors= math.floor(test_size* m_ins)

    knn= NearestNeighbors(n_neighbors= neighbors)
    knn.fit(df_X)

    knn_index= knn.kneighbors(target_X, return_distance=False)
    
    trainX= df_X.drop(df_X.index[knn_index[0]])
    trainY= df_Y.drop(df_Y.index[knn_index[0]])

    testX= df_X.loc[df_X.index[knn_index[0]]]
    testY= df_Y.loc[df_Y.index[knn_index[0]]]

    testX= pd.concat([target_X, testX])
    testY= pd.concat([target_Y, testY])
    
    return trainX, testX, trainY, testY

In [20]:
from sklearn.model_selection import KFold, StratifiedKFold

# df_X and df_Y doesn't contain target_X and target_Y
def knnfold_remove_and_retrain(model, df_X, df_Y, target_X, target_Y, numeric_columns, num_type='mean', cat_type='none', test_size=0.01):

    m_ins= df_X.shape[0]
    n_fts= df_X.shape[1]
    
    k_viz= math.floor(test_size* m_ins)
    k_viz_aux= math.floor(np.sqrt(m_ins))
    
    if (k_viz< k_viz_aux):
        k_viz= k_viz_aux
        
    # here, we generate a neighborhood of the target instance, our test set
    x_train, x_test, y_train, y_test= knn_train_test_split(df_X, df_Y, target_X, target_Y, 
                                                                         test_size= (k_viz/ m_ins))    
    K_folds= math.floor(np.sqrt(n_fts))

    if (K_folds< 5):
        K_folds= 5    

    skf= StratifiedKFold(n_splits= K_folds, random_state=1234, shuffle=True)    
        
    acc_all= []
    acc_no_i= np.zeros(n_fts)
    acc_no_ij= np.zeros((n_fts,n_fts))
    
    for train_index, test_index in skf.split(x_train, y_train):
        
        x_t_fold= x_train.iloc[train_index]
        y_t_fold= y_train.iloc[train_index]
        
        acc_all.append(train_model_get_acc_mean(model, x_t_fold, x_test, y_t_fold, y_test, 1))
        
        replace_ft= replace_values(x_t_fold, numeric_columns, num_type=num_type, cat_type=cat_type)
        
        aux_acc_no_i, aux_acc_no_ij= remove_and_retrain_v1(model, replace_ft, x_t_fold, x_test, 
                                                           y_t_fold, y_test, 1)
        
        acc_no_i += aux_acc_no_i
        acc_no_ij += aux_acc_no_ij

    
    acc_no_i /= K_folds
    acc_no_ij /= K_folds

    mean_acc_all= np.mean(acc_all)
    
    return mean_acc_all, acc_no_i, acc_no_ij

# Feature Attribution using PageRank methods

In [21]:
def run_lib_pr(graph_matrix, ft_names):
    
    num_fts= graph_matrix.shape[0]
    
    D= nx.DiGraph()

    for i in range(num_fts):
        for j in range(num_fts):
            if (i!= j):
                D.add_weighted_edges_from([(ft_names[i],ft_names[j],graph_matrix[i,j])])
                
    pRank= pd.Series(nx.pagerank(D, max_iter=100, alpha=0.85, tol=1.0e-6))

    return pRank.sort_values(ascending=False)

In [22]:
# using my PR to compare results
def run_my_pr(graph_matrix, ft_names):
    
    file_path= 'datasets/FAR_data.txt'
    
    num_fts= graph_matrix.shape[0]

    f= open(file_path, 'w')

    for i in range(num_fts):
        for j in range(num_fts):
            line= (str(int(i)) + ',' + str(int(j)) + ',' + str(graph_matrix[i,j]) + '\n')
            f.write(line)

    f.close()

    graph= gpr.init_graph(file_path)

    myPRank= pd.Series(pr.run_PageRank(graph, iteration= 100, damping_factor= 0.95, tolerance= 1.0e-6))
    myPRank= myPRank.sort_values(ascending=False)
    
    ids_names= ft_names[(myPRank.index).astype(int)]

    myPRank.index= ids_names

    return myPRank

# --- Tests using the Titanic dataset ---

# Data loading and preprocessing

In [23]:
!kaggle competitions download -c titanic

/bin/bash: kaggle: command not found


In [24]:
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

train_data.shape, test_data.shape

((891, 12), (418, 11))

In [25]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [26]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [27]:
X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

X_all

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
1305,3,male,NaN,0,0,8.0500,S
1306,1,female,39.0,0,0,108.9000,C
1307,3,male,38.5,0,0,7.2500,S


In [28]:
numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [29]:
# in this case we'll only use X_train df because X_test is not labeled

In [30]:
X_train= pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')

X_train= pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
887,2,male,27.0,0,0,13.0000,S
888,1,female,19.0,0,0,30.0000,S
889,3,female,28.0,1,2,23.4500,S


In [31]:
# one-hot encoding the qualitative features
X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train_ohe.head()

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
PassengerId,,,,,,,,,,,,
1,22.0,1,0,7.2500,0,0,1,0,1,0,0,1
2,38.0,1,0,71.2833,1,0,0,1,0,1,0,0
3,26.0,0,0,7.9250,0,0,1,1,0,0,0,1
4,35.0,1,0,53.1000,1,0,0,1,0,0,0,1
5,35.0,0,0,8.0500,0,0,1,0,1,0,0,1


In [32]:
y_train.head()

PassengerId
1    0
2    1
3    1
4    1
5    0
Name: Survived, dtype: int64

In [33]:
# normalize the numeric columns of dataframe with each value between 0 and 1
X_train= normalize_selected_cols(X_train, numeric_columns)
X_train_ohe= normalize_selected_cols(X_train_ohe, numeric_columns)

# ML model setup

In [34]:
train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)

#train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
#xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)

#train, test, labels_train, labels_test= train_test_split(X_train,y_train,train_size=0.80,random_state=1234)
#cat_fts= [train.columns.to_list().index(col) for col in categor_columns]
#ctb_model= CatBoostClassifier(cat_features=cat_fts,silent=True)

# --- Run global modeling ---

In [35]:
# re-training can result in slightly different models, it is essential to repeat the training process multiple times to ensure that the variance in accuracy is low 
# (Hooker et al., 2018)
repeat_train= 1

num_fts= len(train.columns)

In [36]:
# train the ML model and get the mean accuracy using the entire feature set
acc_all_fts= train_model_get_acc_mean(rf, train, test, labels_train, labels_test, repeat_train)

acc_all_fts

0.8212290502793296

In [37]:
# values to "remove" and retrain
replace_ft= replace_values(train,numeric_columns,num_type='mean',cat_type='median')

replace_ft

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.362142,0.064782,0.064841,0.064072,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


In [38]:
# train the ML model and get the mean accuracy removing and retraining columns from the feature set
acc_no_i, acc_no_ij= remove_and_retrain_v1(rf, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [39]:
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

[0.788 0.816 0.821 0.832 0.827 0.816 0.821 0.821 0.821 0.816 0.821 0.816]
-----------------------------
[[0.    0.804 0.832 0.816 0.799 0.793 0.793 0.804 0.804 0.788 0.793 0.793]
 [0.81  0.    0.821 0.793 0.821 0.821 0.816 0.81  0.81  0.816 0.81  0.816]
 [0.832 0.821 0.    0.821 0.827 0.821 0.821 0.816 0.821 0.827 0.821 0.827]
 [0.804 0.799 0.827 0.    0.838 0.832 0.832 0.832 0.832 0.827 0.821 0.838]
 [0.799 0.821 0.832 0.832 0.    0.827 0.827 0.821 0.827 0.827 0.827 0.816]
 [0.799 0.821 0.827 0.832 0.827 0.    0.788 0.821 0.821 0.821 0.821 0.821]
 [0.793 0.821 0.816 0.832 0.827 0.799 0.    0.821 0.816 0.821 0.821 0.832]
 [0.804 0.81  0.821 0.832 0.821 0.821 0.821 0.    0.687 0.816 0.821 0.81 ]
 [0.804 0.81  0.827 0.832 0.821 0.821 0.821 0.693 0.    0.816 0.821 0.821]
 [0.793 0.81  0.827 0.821 0.827 0.827 0.821 0.821 0.816 0.    0.816 0.821]
 [0.788 0.81  0.816 0.821 0.821 0.821 0.821 0.821 0.821 0.816 0.    0.816]
 [0.799 0.81  0.832 0.821 0.816 0.821 0.816 0.821 0.821 0.816 0.81  0. 

In [40]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.017 0.045 0.028 0.011 0.006 0.006 0.017 0.017 0.    0.006 0.006]
 [0.006 0.    0.006 0.022 0.006 0.006 0.    0.006 0.006 0.    0.006 0.   ]
 [0.011 0.    0.    0.    0.006 0.    0.    0.006 0.    0.006 0.    0.006]
 [0.028 0.034 0.006 0.    0.006 0.    0.    0.    0.    0.006 0.011 0.006]
 [0.028 0.006 0.006 0.006 0.    0.    0.    0.006 0.    0.    0.    0.011]
 [0.017 0.006 0.011 0.017 0.011 0.    0.028 0.006 0.006 0.006 0.006 0.006]
 [0.028 0.    0.006 0.011 0.006 0.022 0.    0.    0.006 0.    0.    0.011]
 [0.017 0.011 0.    0.011 0.    0.    0.    0.    0.134 0.006 0.    0.011]
 [0.017 0.011 0.006 0.011 0.    0.    0.    0.128 0.    0.006 0.    0.   ]
 [0.022 0.006 0.011 0.006 0.011 0.011 0.006 0.006 0.    0.    0.    0.006]
 [0.034 0.011 0.006 0.    0.    0.    0.    0.    0.    0.006 0.    0.006]
 [0.017 0.006 0.017 0.006 0.    0.006 0.    0.006 0.006 0.    0.006 0.   ]]
-----------------------------
[[0.    0.011 0.011 0.017 0.028 0.022 0.028 0.017 0.017 0.028 0.028 0

In [41]:
# get the stationary distribution
stationary_d1= p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= p_matrix_to_stationary_dist(p_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.17337508 0.09241306 0.09874609 0.10130065 0.05164467 0.02792659
 0.01525607 0.1643482  0.14996051 0.03622372 0.0325303  0.05627505]
-----------------------------
[0.1266806  0.06601526 0.06208162 0.07653154 0.05280299 0.09233576
 0.08156793 0.12948565 0.139615   0.05304061 0.0434998  0.07634323]
-----------------------------
[0.11537254 0.06536805 0.06092407 0.06979553 0.05462231 0.08032486
 0.05955636 0.16223155 0.16900739 0.04965384 0.0429647  0.07017879]
-----------------------------
[0.25669182 0.09880126 0.08872049 0.13392176 0.06782079 0.05355776
 0.01724258 0.07567814 0.06981944 0.04843034 0.02547844 0.06383718]


In [42]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [43]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.173375
Sex_female    0.164348
Sex_male      0.149961
Fare          0.101301
Parch         0.098746
SibSp         0.092413
Embarked_S    0.056275
Pclass_1      0.051645
Embarked_C    0.036224
Embarked_Q    0.032530
Pclass_2      0.027927
Pclass_3      0.015256
dtype: float64

In [44]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.314309
1,Age,0.173375
2,Embarked,0.125029
3,Fare,0.101301
4,Parch,0.098746
5,Pclass,0.094827
6,SibSp,0.092413


In [45]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_male      0.139615
Sex_female    0.129486
Age           0.126681
Pclass_2      0.092336
Pclass_3      0.081568
Fare          0.076532
Embarked_S    0.076343
SibSp         0.066015
Parch         0.062082
Embarked_C    0.053041
Pclass_1      0.052803
Embarked_Q    0.043500
dtype: float64

In [46]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.269101
1,Pclass,0.226707
2,Embarked,0.172884
3,Age,0.126681
4,Fare,0.076532
5,SibSp,0.066015
6,Parch,0.062082


In [47]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_male      0.169007
Sex_female    0.162232
Age           0.115373
Pclass_2      0.080325
Embarked_S    0.070179
Fare          0.069796
SibSp         0.065368
Parch         0.060924
Pclass_3      0.059556
Pclass_1      0.054622
Embarked_C    0.049654
Embarked_Q    0.042965
dtype: float64

In [48]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.314309
1,Age,0.173375
2,Embarked,0.125029
3,Fare,0.101301
4,Parch,0.098746
5,Pclass,0.094827
6,SibSp,0.092413


In [49]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.256692
Fare          0.133922
SibSp         0.098801
Parch         0.088720
Sex_female    0.075678
Sex_male      0.069819
Pclass_1      0.067821
Embarked_S    0.063837
Pclass_2      0.053558
Embarked_C    0.048430
Embarked_Q    0.025478
Pclass_3      0.017243
dtype: float64

In [50]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.314309
1,Age,0.173375
2,Embarked,0.125029
3,Fare,0.101301
4,Parch,0.098746
5,Pclass,0.094827
6,SibSp,0.092413


# --- Run local modeling ---

In [51]:
X_to_split= X_train_ohe.copy()
Y_to_split= y_train.copy()

In [52]:
target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [53]:
Y_no_targt.head()

PassengerId
2    1
3    1
4    1
5    0
6    0
Name: Survived, dtype: int64

In [54]:
num_fts= len(X_no_targt.columns)

acc_all_fts, acc_no_i, acc_no_ij= knnfold_remove_and_retrain(rf, X_no_targt, Y_no_targt, target_instance, 
                                                              target_label, numeric_columns, num_type='mean', 
                                                              cat_type='median', test_size= 0.2)

In [55]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8815642458100559
-----------------------------
[0.872 0.88  0.883 0.384 0.883 0.88  0.883 0.883 0.883 0.882 0.882 0.882]
-----------------------------
[[0.    0.876 0.83  0.873 0.873 0.868 0.869 0.872 0.868 0.872 0.868 0.872]
 [0.879 0.    0.877 0.352 0.883 0.88  0.88  0.88  0.88  0.882 0.88  0.88 ]
 [0.83  0.877 0.    0.385 0.882 0.88  0.882 0.882 0.882 0.883 0.882 0.882]
 [0.875 0.361 0.331 0.    0.287 0.437 0.398 0.266 0.321 0.25  0.421 0.468]
 [0.87  0.883 0.882 0.236 0.    0.883 0.88  0.883 0.883 0.883 0.883 0.883]
 [0.868 0.88  0.882 0.398 0.883 0.    0.875 0.88  0.883 0.883 0.88  0.883]
 [0.869 0.88  0.88  0.454 0.88  0.873 0.    0.883 0.883 0.883 0.882 0.883]
 [0.87  0.88  0.882 0.323 0.883 0.88  0.883 0.    0.666 0.883 0.882 0.883]
 [0.87  0.88  0.883 0.312 0.883 0.88  0.882 0.667 0.    0.882 0.883 0.882]
 [0.873 0.882 0.882 0.355 0.882 0.882 0.883 0.883 0.883 0.    0.883 0.801]
 [0.869 0.88  0.88  0.468 0.883 0.88  0.88  0.882 0.883 0.883 0.    0.878]
 [0.87  0.88  0.882 0.

In [56]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.004 0.041 0.001 0.001 0.003 0.002 0.    0.003 0.    0.003 0.   ]
 [0.001 0.    0.003 0.528 0.002 0.    0.    0.    0.    0.001 0.    0.   ]
 [0.053 0.006 0.    0.497 0.001 0.002 0.001 0.001 0.001 0.    0.001 0.001]
 [0.491 0.023 0.054 0.    0.097 0.053 0.013 0.118 0.064 0.134 0.037 0.084]
 [0.012 0.    0.001 0.647 0.    0.    0.002 0.    0.    0.    0.    0.   ]
 [0.012 0.    0.001 0.483 0.002 0.    0.006 0.    0.002 0.002 0.    0.002]
 [0.013 0.002 0.002 0.429 0.002 0.01  0.    0.    0.    0.    0.001 0.   ]
 [0.012 0.002 0.001 0.56  0.    0.002 0.    0.    0.217 0.    0.001 0.   ]
 [0.012 0.002 0.    0.571 0.    0.002 0.001 0.216 0.    0.001 0.    0.001]
 [0.009 0.    0.    0.526 0.    0.    0.001 0.001 0.001 0.    0.001 0.08 ]
 [0.012 0.001 0.001 0.413 0.001 0.001 0.001 0.    0.001 0.001 0.    0.003]
 [0.011 0.001 0.    0.455 0.001 0.001 0.    0.    0.001 0.069 0.004 0.   ]]
-----------------------------
[[0.    0.004 0.053 0.488 0.01  0.012 0.013 0.011 0.015 0.01  0.013 0

In [57]:
# get the stationary distribution
stationary_d1= p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= p_matrix_to_stationary_dist(p_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.17839653 0.02271588 0.14019081 0.3810881  0.0357127  0.02826321
 0.01189862 0.05111437 0.04524171 0.04841593 0.02269227 0.03426987]
-----------------------------
[0.04598282 0.03182369 0.04146525 0.27926902 0.03495801 0.02918171
 0.03868797 0.14606486 0.14495662 0.08805993 0.03188213 0.08766799]
-----------------------------
[0.05530806 0.03380083 0.04235307 0.27389149 0.03546141 0.03315511
 0.04059909 0.13013136 0.12962896 0.09264588 0.0365205  0.09650425]
-----------------------------
[0.20633847 0.0125375  0.03771846 0.47113891 0.04036334 0.02360683
 0.00765313 0.05430459 0.03686422 0.05584986 0.0161842  0.03744049]


In [58]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [59]:
# TARGET INSTANCE TO EXPLAIN ITS FEATURES

X_all.loc[X_all.index== target_index]

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.25,S


In [60]:
target_label

PassengerId
1    0
Name: Survived, dtype: int64

In [61]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.381088
Age           0.178397
Parch         0.140191
Sex_female    0.051114
Embarked_C    0.048416
Sex_male      0.045242
Pclass_1      0.035713
Embarked_S    0.034270
Pclass_2      0.028263
SibSp         0.022716
Embarked_Q    0.022692
Pclass_3      0.011899
dtype: float64

In [62]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.381088
1,Age,0.178397
2,Parch,0.140191
3,Embarked,0.105378
4,Sex,0.096356
5,Pclass,0.075875
6,SibSp,0.022716


In [63]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.279269
Sex_female    0.146065
Sex_male      0.144957
Embarked_C    0.088060
Embarked_S    0.087668
Age           0.045983
Parch         0.041465
Pclass_3      0.038688
Pclass_1      0.034958
Embarked_Q    0.031882
SibSp         0.031824
Pclass_2      0.029182
dtype: float64

In [64]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.291021
1,Fare,0.279269
2,Embarked,0.207610
3,Pclass,0.102828
4,Age,0.045983
5,Parch,0.041465
6,SibSp,0.031824


In [65]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.273891
Sex_female    0.130131
Sex_male      0.129629
Embarked_S    0.096504
Embarked_C    0.092646
Age           0.055308
Parch         0.042353
Pclass_3      0.040599
Embarked_Q    0.036521
Pclass_1      0.035461
SibSp         0.033801
Pclass_2      0.033155
dtype: float64

In [66]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.381088
1,Age,0.178397
2,Parch,0.140191
3,Embarked,0.105378
4,Sex,0.096356
5,Pclass,0.075875
6,SibSp,0.022716


In [67]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.471139
Age           0.206338
Embarked_C    0.055850
Sex_female    0.054305
Pclass_1      0.040363
Parch         0.037718
Embarked_S    0.037440
Sex_male      0.036864
Pclass_2      0.023607
Embarked_Q    0.016184
SibSp         0.012537
Pclass_3      0.007653
dtype: float64

In [68]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.381088
1,Age,0.178397
2,Parch,0.140191
3,Embarked,0.105378
4,Sex,0.096356
5,Pclass,0.075875
6,SibSp,0.022716


# --- Run PageRank approach ---

In [69]:
X_to_split= X_train_ohe.copy()
Y_to_split= y_train.copy()

target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [70]:
num_fts= len(X_no_targt.columns)

acc_all_fts, acc_no_i, acc_no_ij= knnfold_remove_and_retrain(rf, X_no_targt, Y_no_targt, target_instance, 
                                                              target_label, numeric_columns, num_type='mean', 
                                                              cat_type='median', test_size= 0.2)

In [71]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8826815642458101
-----------------------------
[0.873 0.882 0.882 0.304 0.883 0.882 0.882 0.883 0.883 0.883 0.883 0.883]
-----------------------------
[[0.    0.879 0.829 0.873 0.873 0.87  0.869 0.872 0.868 0.87  0.868 0.869]
 [0.876 0.    0.877 0.356 0.883 0.88  0.88  0.88  0.88  0.883 0.88  0.88 ]
 [0.83  0.877 0.    0.333 0.883 0.88  0.882 0.882 0.882 0.88  0.88  0.882]
 [0.873 0.368 0.333 0.    0.238 0.388 0.41  0.364 0.321 0.296 0.42  0.486]
 [0.872 0.883 0.883 0.277 0.    0.883 0.879 0.883 0.883 0.882 0.883 0.883]
 [0.869 0.88  0.88  0.456 0.883 0.    0.872 0.882 0.882 0.883 0.882 0.883]
 [0.868 0.88  0.88  0.396 0.878 0.87  0.    0.883 0.883 0.883 0.882 0.882]
 [0.868 0.88  0.88  0.273 0.883 0.883 0.882 0.    0.664 0.883 0.882 0.882]
 [0.869 0.88  0.88  0.407 0.883 0.882 0.882 0.666 0.    0.883 0.883 0.883]
 [0.873 0.882 0.882 0.378 0.883 0.882 0.883 0.883 0.883 0.    0.883 0.782]
 [0.868 0.88  0.88  0.468 0.883 0.88  0.882 0.882 0.883 0.882 0.    0.878]
 [0.869 0.88  0.882 0.

In [72]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= p_matrix_to_row_stochastic_matrix(p_matrix1)
st_matrix2= p_matrix_to_row_stochastic_matrix(p_matrix2)
st_matrix3= p_matrix_to_row_stochastic_matrix(p_matrix3)
st_matrix4= p_matrix_to_row_stochastic_matrix(p_matrix4)

In [73]:
run_lib_pr(st_matrix1, X_train_ohe.columns)

Fare          0.362660
Age           0.161742
Parch         0.104170
Embarked_S    0.066771
Embarked_Q    0.049622
Pclass_3      0.045340
SibSp         0.041965
Pclass_2      0.038262
Sex_female    0.037902
Sex_male      0.033626
Embarked_C    0.029249
Pclass_1      0.028689
dtype: float64

In [74]:
run_lib_pr(st_matrix2, X_train_ohe.columns)

Fare          0.314005
Sex_male      0.124793
Sex_female    0.112588
Embarked_S    0.069185
Embarked_C    0.066246
Parch         0.048560
Pclass_1      0.047513
Pclass_2      0.045248
SibSp         0.044315
Pclass_3      0.043139
Age           0.042713
Embarked_Q    0.041695
dtype: float64

In [75]:
run_lib_pr(st_matrix3, X_train_ohe.columns)

Fare          0.307166
Sex_male      0.122709
Sex_female    0.111399
Embarked_S    0.072193
Embarked_C    0.065198
Age           0.053672
Parch         0.047177
Pclass_2      0.045462
Pclass_1      0.044040
Pclass_3      0.044010
SibSp         0.043721
Embarked_Q    0.043254
dtype: float64

In [76]:
run_lib_pr(st_matrix4, X_train_ohe.columns)

Fare          0.442669
Age           0.185140
Embarked_S    0.067102
Embarked_Q    0.047236
Pclass_3      0.045087
Pclass_2      0.038544
SibSp         0.033669
Sex_female    0.033356
Parch         0.032576
Pclass_1      0.031566
Sex_male      0.022790
Embarked_C    0.020266
dtype: float64

In [77]:
run_my_pr(st_matrix1, X_train_ohe.columns)

Fare          0.322145
Age           0.171838
SibSp         0.059070
Parch         0.058588
Sex_female    0.056602
Embarked_C    0.051347
Embarked_S    0.048463
Embarked_Q    0.047185
Pclass_2      0.046947
Pclass_3      0.046664
Sex_male      0.045963
Pclass_1      0.045186
dtype: float64

In [78]:
run_my_pr(st_matrix2, X_train_ohe.columns)

Fare          0.289873
Age           0.091971
Sex_female    0.082350
SibSp         0.078902
Parch         0.077697
Embarked_C    0.065965
Sex_male      0.055283
Pclass_2      0.054315
Embarked_S    0.052000
Pclass_1      0.051327
Embarked_Q    0.050448
Pclass_3      0.049869
dtype: float64

In [79]:
run_my_pr(st_matrix3, X_train_ohe.columns)

Fare          0.283295
Age           0.100879
Sex_female    0.082262
SibSp         0.078048
Parch         0.075654
Embarked_C    0.065908
Sex_male      0.055189
Pclass_2      0.054334
Embarked_S    0.052317
Pclass_1      0.051536
Embarked_Q    0.050594
Pclass_3      0.049984
dtype: float64

In [80]:
run_my_pr(st_matrix4, X_train_ohe.columns)

Fare          0.341798
Age           0.176010
SibSp         0.058264
Parch         0.051423
Sex_female    0.050726
Embarked_S    0.047493
Embarked_C    0.046896
Embarked_Q    0.046069
Pclass_3      0.046028
Pclass_2      0.046001
Pclass_1      0.044859
Sex_male      0.044435
dtype: float64

# --- Using knn to reduce the dataset ---

In [81]:
target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [82]:
train, test, labels_train, labels_test= knn_train_test_split(X_no_targt,Y_no_targt,
                                                            target_instance,target_label,test_size=0.5)

In [83]:
# here we'll only consider the test set from knn split because this set contains the test_size percentage 
# of nearest elements from the target instance
# target_X and target_Y is into test_X and test_Y, we drop the target instance again

new_X_no_tgt= test.drop(target_instance.index)
new_Y_no_tgt= labels_test.drop(index= target_index)

In [84]:
repeat_train= 1
num_fts= len(train.columns)

start= time.time()

# Feature Attribution using Raking - Knn-Local - Remove and Retrain
train_1, test_1, labels_train_1, labels_test_1= knn_train_test_split(new_X_no_tgt,new_Y_no_tgt,
                                                                     target_instance,target_label,
                                                                     test_size=0.2)

acc_all_fts= train_model_get_acc_mean(rf, train_1, test_1, labels_train_1.values.ravel(), 
                                      labels_test_1.values.ravel(), repeat_train)

replace_ft= replace_values(train_1,train_1.columns,num_type='mean',cat_type='median')

acc_no_i, acc_no_ij= remove_and_retrain_v1(rf, replace_ft, train_1, test_1, labels_train_1,
                                           labels_test_1, repeat_train)

end= time.time()
print("--- %s seconds ---" % np.round((end- start), 2))

--- 125.1 seconds ---


In [85]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8888888888888888
-----------------------------
[0.878 0.867 0.878 0.6   0.878 0.911 0.867 0.867 0.878 0.878 0.9   0.9  ]
-----------------------------
[[0.    0.911 0.811 0.922 0.878 0.878 0.878 0.878 0.878 0.878 0.878 0.878]
 [0.911 0.    0.9   0.522 0.867 0.867 0.867 0.867 0.867 0.867 0.889 0.889]
 [0.811 0.911 0.    0.6   0.878 0.878 0.878 0.878 0.878 0.9   0.9   0.9  ]
 [0.922 0.522 0.722 0.    0.6   0.6   0.722 0.6   0.722 0.6   0.722 0.722]
 [0.878 0.867 0.878 0.722 0.    0.9   0.889 0.844 0.878 0.878 0.878 0.9  ]
 [0.878 0.889 0.867 0.722 0.878 0.    0.922 0.867 0.911 0.9   0.878 0.9  ]
 [0.878 0.867 0.878 0.722 0.867 0.922 0.    0.889 0.878 0.878 0.878 0.9  ]
 [0.878 0.856 0.867 0.6   0.878 0.867 0.878 0.    0.6   0.856 0.889 0.9  ]
 [0.878 0.867 0.878 0.6   0.856 0.856 0.856 0.589 0.    0.867 0.867 0.9  ]
 [0.878 0.867 0.878 0.6   0.9   0.889 0.878 0.889 0.856 0.    0.867 0.878]
 [0.878 0.867 0.9   0.722 0.9   0.9   0.911 0.878 0.878 0.878 0.    0.922]
 [0.878 0.889 0.9   0.

In [86]:
start= time.time()

# Feature Attribution using Raking - Local - knn and Kfold Remove and Retrain
acc_all_fts, acc_no_i_k, acc_no_ij_k= knnfold_remove_and_retrain(rf,new_X_no_tgt,new_Y_no_tgt,
                                                                target_instance,target_label,numeric_columns, 
                                                                num_type='mean',cat_type='median', 
                                                                test_size= 0.2)

end= time.time()
print("--- %s seconds ---" % np.round((end- start), 2))

--- 634.19 seconds ---


In [87]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i_k, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij_k, decimals=3))

0.888888888888889
-----------------------------
[0.871 0.88  0.884 0.624 0.88  0.891 0.889 0.86  0.862 0.878 0.884 0.896]
-----------------------------
[[0.    0.889 0.82  0.922 0.871 0.871 0.871 0.871 0.867 0.871 0.871 0.871]
 [0.889 0.    0.891 0.58  0.88  0.871 0.876 0.858 0.858 0.876 0.88  0.882]
 [0.818 0.893 0.    0.649 0.878 0.884 0.873 0.873 0.869 0.869 0.898 0.893]
 [0.922 0.58  0.649 0.    0.647 0.678 0.698 0.624 0.649 0.649 0.649 0.669]
 [0.871 0.86  0.893 0.647 0.    0.887 0.88  0.871 0.867 0.88  0.887 0.896]
 [0.871 0.878 0.898 0.702 0.887 0.    0.902 0.871 0.873 0.893 0.893 0.896]
 [0.871 0.878 0.893 0.624 0.882 0.909 0.    0.862 0.871 0.887 0.88  0.898]
 [0.871 0.847 0.867 0.649 0.867 0.869 0.867 0.    0.602 0.858 0.878 0.893]
 [0.867 0.864 0.86  0.624 0.86  0.862 0.86  0.596 0.    0.853 0.869 0.889]
 [0.867 0.867 0.878 0.622 0.88  0.878 0.88  0.86  0.853 0.    0.891 0.856]
 [0.871 0.869 0.891 0.656 0.884 0.893 0.891 0.88  0.878 0.862 0.    0.911]
 [0.871 0.884 0.896 0.6

# Misc

In [88]:
eigval, eigvec= eigs(st_matrix3.T, k=1, which='LM')
eigval

array([1.+0.j])

In [89]:
f= open('datasets/FAR_data.txt', 'w')

for i in range(num_fts):
    for j in range(num_fts):
        line= (str(i) + ',' + str(j) + ',' + str(p_matrix1[i][j]) + '\n')
        f.write(line)
        
f.close()